# NBA DFS Walk-Forward Backtest - Model Training & Prediction

This notebook runs the walk-forward backtesting pipeline:
1. Load historical data for training
2. Build features using YAML-configured pipelines
3. Train XGBoost models (per-player or slate-level)
4. Generate predictions for each test slate
5. Save predictions and models to outputs directory

Results are saved to `data/outputs/{timestamp}/predictions/` for later evaluation.

## Configuration Options

### GPU Acceleration
- Set `USE_GPU = True` to enable GPU training
- Configure `MODEL_CONFIG_PATH` for GPU-optimized hyperparameters

### Player Filtering
- **Salary**: `FILTER_SALARY_MIN`, `FILTER_SALARY_MAX`
- **Injury**: `FILTER_EXCLUDE_OUT`, `FILTER_EXCLUDE_DOUBTFUL`, `FILTER_EXCLUDE_QUESTIONABLE`
- **Players**: `FILTER_PLAYER_IDS`, `FILTER_PLAYER_NAMES`, `FILTER_PLAYERS_CSV`

## Next Step
After running this notebook, use `evaluate_backtest.ipynb` to analyze results and generate visualizations.

## Setup

In [ ]:
import sys
from pathlib import Path
import logging
import pandas as pd

from datetime import datetime, timedelta

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluation.backtest import WalkForwardBacktest
from src.data.loaders.historical_loader import HistoricalDataLoader

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print('Setup complete')

## System Resources

In [2]:
import psutil
import multiprocessing

cpu_count = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / (1024**3)

print(f"CPU Cores: {cpu_count}")
print(f"RAM: {ram_gb:.1f} GB")
print(f"Recommended n_jobs: {cpu_count}")

if ram_gb < 12:
    print("WARNING: Low RAM detected. Consider reducing n_jobs or processing fewer players.")

CPU Cores: 32
RAM: 31.8 GB
Recommended n_jobs: 32


## Configuration

In [ ]:
# Paths
from src.config.paths import OUTPUTS_DIR, PROJECT_ROOT, DATA_DIR
OUTPUT_DIR = str(OUTPUTS_DIR)
DATA_DIR_PATH = str(DATA_DIR)

TRAIN_START = 20241001 
TRAIN_END = 20250201
# Date ranges
TEST_START = 20250205
TEST_END = 20250430

# Model configuration
NUM_SEASONS = 1
FEATURE_CONFIG = 'opponent_features'
MODEL_TYPE = 'xgboost'
MIN_PLAYER_GAMES = 10
MIN_GAMES_FOR_BENCHMARK = 5
RECALIBRATE_DAYS = 7
SALARY_TIERS = [0, 4000, 6000, 8000, 15000]

# Execution settings
PER_PLAYER_MODELS = False
SAVE_MODELS = True
SAVE_PREDICTIONS = True
N_JOBS = 32

# GPU configuration
USE_GPU = False
GPU_ID = 0
MODEL_CONFIG_PATH = "C:\\Users\\antho\\OneDrive\\Documents\\Repositories\\delapan-fantasy\\config\\models\\xgboost_default.yaml"

# Player filtering - Salary and Injury
FILTER_SALARY_MIN = 5000
FILTER_SALARY_MAX = None
FILTER_EXCLUDE_OUT = False
FILTER_EXCLUDE_DOUBTFUL = False
FILTER_EXCLUDE_QUESTIONABLE = False

# Player filtering - ID and Name
FILTER_PLAYER_IDS = None 
FILTER_PLAYER_NAMES = []
#FILTER_PLAYER_NAMES = ['LeBron James', 'Stephen Curry', 'Nikola Jokic', 'Kevin Durant', 'Devin Booker']  # Fixed spelling: LeBron with capital B
FILTER_PLAYERS_CSV = None


# Load model parameters
if USE_GPU and MODEL_CONFIG_PATH:
    import yaml
    with open(repo_root / MODEL_CONFIG_PATH, 'r') as f:
        gpu_config = yaml.safe_load(f)
        MODEL_PARAMS = gpu_config.get('model', {}).get('params', {})
        if 'device' not in MODEL_PARAMS:
            MODEL_PARAMS['device'] = f'cuda:{GPU_ID}'
        if 'tree_method' not in MODEL_PARAMS:
            MODEL_PARAMS['tree_method'] = 'hist'
else:
    MODEL_PARAMS = {
        'max_depth': 6,
        'learning_rate': 0.05,
        'n_estimators': 200,
        'min_child_weight': 5,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'objective': 'reg:squarederror',
        'random_state': 42
    }
    if USE_GPU:
        MODEL_PARAMS['device'] = f'cuda:{GPU_ID}'
        MODEL_PARAMS['tree_method'] = 'hist'

# Display configuration
print('Configuration:')
print(f'  Data Directory: {DATA_DIR_PATH}')
print(f'  Output Directory: {OUTPUT_DIR}')
print(f'  Training Period: {TRAIN_START} to {TRAIN_END}')
print(f'  Testing Period: {TEST_START} to {TEST_END}')
print(f'  Number of Seasons: {NUM_SEASONS}')
print(f'  Model Type: {MODEL_TYPE}')
print(f'  Feature Config: {FEATURE_CONFIG}')
print(f'  Per-Player Models: {PER_PLAYER_MODELS}')
print(f'  Min Player Games: {MIN_PLAYER_GAMES}')
print(f'  Min Benchmark Games: {MIN_GAMES_FOR_BENCHMARK}')
print(f'  Recalibrate Every: {RECALIBRATE_DAYS} days')
print(f'  Parallel Jobs: {N_JOBS}')
print(f'  Save Models: {SAVE_MODELS}')
print(f'  Save Predictions: {SAVE_PREDICTIONS}')
print(f'  Salary Tiers: {SALARY_TIERS}')

if USE_GPU:
    print(f'\n  GPU Configuration:')
    print(f'    Enabled: Yes')
    print(f'    GPU ID: {GPU_ID}')
    print(f'    Device: {MODEL_PARAMS.get("device", "N/A")}')
    print(f'    Tree Method: {MODEL_PARAMS.get("tree_method", "N/A")}')
    if MODEL_CONFIG_PATH:
        print(f'    Config File: {MODEL_CONFIG_PATH}')

print(f'\n  Player Filters:')
if FILTER_SALARY_MIN:
    print(f'    - Minimum salary: ${FILTER_SALARY_MIN}')
if FILTER_SALARY_MAX:
    print(f'    - Maximum salary: ${FILTER_SALARY_MAX}')
if FILTER_PLAYER_NAMES:
    names_list = FILTER_PLAYER_NAMES if isinstance(FILTER_PLAYER_NAMES, list) else [FILTER_PLAYER_NAMES]
    names_display = ', '.join(names_list[:2])
    if len(names_list) > 2:
        names_display += f", ... (+{len(names_list)-2} more)"
    print(f'    - Filter by player names: {names_display}')
if FILTER_PLAYER_IDS:
    print(f'    - Filter by player IDs: {FILTER_PLAYER_IDS}')
if FILTER_PLAYERS_CSV:
    print(f'    - Filter from CSV file: {FILTER_PLAYERS_CSV}')

## Run Backtest

Execute the walk-forward backtesting pipeline. This will:
- Load and process training data
- Build feature pipelines
- Train models for each slate
- Generate predictions
- Save all outputs to timestamped directory

In [ ]:
from src.filters import ColumnFilter, InjuryFilter
from src.filters.player_filters import PlayerIDFilter, PlayerNameFilter, PlayerIDFromCSVFilter

# Build player filters
player_filters = []

if FILTER_SALARY_MIN is not None:
    player_filters.append(ColumnFilter('salary', '>=', FILTER_SALARY_MIN))
    print(f'Added filter: salary >= {FILTER_SALARY_MIN}')

if FILTER_SALARY_MAX is not None:
    player_filters.append(ColumnFilter('salary', '<=', FILTER_SALARY_MAX))
    print(f'Added filter: salary <= {FILTER_SALARY_MAX}')

if FILTER_EXCLUDE_OUT or FILTER_EXCLUDE_DOUBTFUL or FILTER_EXCLUDE_QUESTIONABLE:
    injury_filter = InjuryFilter(
        exclude_out=FILTER_EXCLUDE_OUT,
        exclude_doubtful=FILTER_EXCLUDE_DOUBTFUL,
        exclude_questionable=FILTER_EXCLUDE_QUESTIONABLE
    )
    player_filters.append(injury_filter)
    excluded = []
    if FILTER_EXCLUDE_OUT:
        excluded.append('OUT')
    if FILTER_EXCLUDE_DOUBTFUL:
        excluded.append('DOUBTFUL')
    if FILTER_EXCLUDE_QUESTIONABLE:
        excluded.append('QUESTIONABLE')
    print(f'Added filter: exclude injury status {", ".join(excluded)}')

if FILTER_PLAYER_IDS:
    if isinstance(FILTER_PLAYER_IDS, str):
        player_ids = [pid.strip() for pid in FILTER_PLAYER_IDS.replace(',', ' ').split() if pid.strip()]
    else:
        player_ids = FILTER_PLAYER_IDS if isinstance(FILTER_PLAYER_IDS, list) else [FILTER_PLAYER_IDS]
    
    if player_ids:
        player_id_filter = PlayerIDFilter(player_ids)
        player_filters.append(player_id_filter)
        print(f'Added filter: player ID in {player_ids}')

if FILTER_PLAYER_NAMES:
    if isinstance(FILTER_PLAYER_NAMES, str):
        player_names = [name.strip() for name in FILTER_PLAYER_NAMES.replace(',', '|').split('|') if name.strip()]
    else:
        player_names = FILTER_PLAYER_NAMES if isinstance(FILTER_PLAYER_NAMES, list) else [FILTER_PLAYER_NAMES]
    
    if player_names:
        player_name_filter = PlayerNameFilter(player_names, case_sensitive=False)
        player_filters.append(player_name_filter)
        print(f'Added filter: player name contains {player_names}')

if FILTER_PLAYERS_CSV:
    try:
        csv_filter = PlayerIDFromCSVFilter(FILTER_PLAYERS_CSV)
        player_filters.append(csv_filter)
        print(f'Added filter: player IDs from CSV ({len(csv_filter.player_ids)} players)')
    except Exception as e:
        print(f'ERROR loading CSV filter: {e}')

if player_filters:
    print(f'\nTotal filters: {len(player_filters)}')
else:
    print('No player filters configured')

# Initialize and run backtest
backtest = WalkForwardBacktest(
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    model_type=MODEL_TYPE,
    model_params=MODEL_PARAMS,
    feature_config=FEATURE_CONFIG,
    output_dir=OUTPUT_DIR,
    data_dir=DATA_DIR_PATH,
    per_player_models=PER_PLAYER_MODELS,
    min_player_games=MIN_PLAYER_GAMES,
    min_games_for_benchmark=MIN_GAMES_FOR_BENCHMARK,
    recalibrate_days=RECALIBRATE_DAYS,
    num_seasons=NUM_SEASONS,
    salary_tiers=SALARY_TIERS,
    save_models=SAVE_MODELS,
    save_predictions=SAVE_PREDICTIONS,
    n_jobs=N_JOBS,
    player_filters=player_filters if player_filters else None,
    benchmark_use_all_history=True  # Benchmark expands from earliest player date
)

print('\n' + '='*80)
print('RUNNING BACKTEST')
print('='*80 + '\n')

results = backtest.run()

if 'error' in results:
    print(f"\nERROR: {results['error']}")
else:
    print(f"\n{'='*80}")
    print('BACKTEST COMPLETED SUCCESSFULLY')
    print('='*80)
    print(f"\nProcessed {results['num_slates']} slates")
    print(f"Model MAPE: {results['model_mean_mape']:.2f}%")
    print(f"Benchmark MAPE: {results['benchmark_mean_mape']:.2f}%")
    print(f"Improvement: {results['mape_improvement']:+.2f}%")


In [7]:
results.keys()

dict_keys(['num_slates', 'date_range', 'model_mean_mape', 'model_median_mape', 'model_std_mape', 'model_mean_cmape', 'model_mean_smape', 'model_mean_wmape', 'model_mean_rmse', 'model_std_rmse', 'model_mean_mae', 'model_mean_correlation', 'model_std_correlation', 'benchmark_mean_mape', 'benchmark_median_mape', 'benchmark_mean_cmape', 'benchmark_mean_wmape', 'mape_improvement', 'total_players_evaluated', 'unique_players_evaluated', 'avg_players_per_slate', 'daily_results', 'all_predictions', 'low_minutes_metrics', 'report_path', 'report_url'])

## Results Summary

In [5]:
if 'error' not in results:
    results_df = results['daily_results']
    all_predictions_df = results['all_predictions']
    
    print('='*80)
    print('BACKTEST RESULTS SUMMARY')
    print('='*80)
    print(f'\nNumber of Slates: {results["num_slates"]}')
    print(f'Date Range: {results["date_range"]}')
    print(f'\nTotal Players Evaluated: {results["total_players_evaluated"]:.0f}')
    print(f'Average Players per Slate: {results["avg_players_per_slate"]:.1f}')
    print(f'\nModel Performance:')
    print(f'  Mean MAPE: {results["model_mean_mape"]:.2f}%')
    print(f'  Median MAPE: {results["model_median_mape"]:.2f}%')
    print(f'  Std MAPE: {results["model_std_mape"]:.2f}%')
    print(f'  Mean RMSE: {results["model_mean_rmse"]:.2f}')
    print(f'  Mean MAE: {results["model_mean_mae"]:.2f}')
    print(f'  Mean Correlation: {results["model_mean_correlation"]:.3f}')
    print(f'\nBenchmark Performance:')
    print(f'  Mean MAPE: {results["benchmark_mean_mape"]:.2f}%')
    print(f'  Median MAPE: {results["benchmark_median_mape"]:.2f}%')
    print(f'\nImprovement (Model vs Benchmark):')
    print(f'  MAPE Improvement: {results["mape_improvement"]:+.2f}%')
    
    if 'statistical_test' in results:
        print(f'\nStatistical Significance:')
        print(f'  p-value: {results["statistical_test"]["p_value"]:.6f}')
        print(f'  Cohen\'s d: {results["statistical_test"]["cohens_d"]:.4f}')
        print(f'  Effect size: {results["statistical_test"]["effect_size"]}')
    
    print(f'\n{'='*80}')
    print('OUTPUT LOCATION')
    print('='*80)
    print(f'\nResults saved to: {backtest.run_output_dir}')
    print(f'\nNext step: Use evaluate_backtest.ipynb to analyze results and generate visualizations')
else:
    print('Backtest failed. Check error message above.')

BACKTEST RESULTS SUMMARY

Number of Slates: 75
Date Range: 20250205 to 20250429

Total Players Evaluated: 8868
Average Players per Slate: 118.2

Model Performance:
  Mean MAPE: 37.17%
  Median MAPE: 36.56%
  Std MAPE: 8.66%
  Mean RMSE: 12.25
  Mean MAE: 9.74
  Mean Correlation: 0.521

Benchmark Performance:
  Mean MAPE: nan%
  Median MAPE: nan%

Improvement (Model vs Benchmark):
  MAPE Improvement: +nan%

OUTPUT LOCATION

Results saved to: c:\Users\antho\OneDrive\Documents\Repositories\delapan-fantasy\data\outputs\20251020_052352

Next step: Use evaluate_backtest.ipynb to analyze results and generate visualizations


## Lineup Optimization

Generate optimal lineups using pydfs-lineup-optimizer based on model predictions.

## Next Steps

### Lineup Optimization Usage

**Contest Types Available:**
- `cash_game`: 50/50s and Double-ups (conservative, floor-focused)
- `gpp_tournament`: Large-field GPPs (aggressive, ceiling-focused)
- `single_entry`: Single-entry tournaments (balanced strategy)
- `multi_entry`: Multi-entry tournaments (diversity-focused)

**Configuration:**
- Modify `LINEUP_CONFIG` to change contest type, number of lineups, target date
- Contest settings are in `config/contests/*.json`
- Export formats: CSV (DraftKings upload), JSON (detailed analysis)

**Workflow:**
1. Run backtest cells above to generate predictions
2. Configure lineup generation settings
3. Generate lineups for target slate
4. Review lineup details and metrics
5. Export for DraftKings upload or further analysis
6. Evaluate lineup performance against actuals (backtesting)

**Advanced Usage:**
- Modify contest configs for custom strategies
- Add player locks/excludes in LineupGenerator
- Implement custom stacking rules
- Track exposure across multiple lineups
- Calculate ROI for different contest types

In [ ]:
# Compare lineup projections with actuals (for backtesting)
if lineups and 'actual_fpts' in slate_df.columns:
    print('='*80)
    print('LINEUP BACKTEST EVALUATION')
    print('='*80)
    
    for i, lineup in enumerate(lineups, 1):
        print(f'\n{"="*60}')
        print(f'LINEUP #{i} - PROJECTED vs ACTUAL')
        print(f'{"="*60}')
        
        total_projected = 0
        total_actual = 0
        
        print(f'\n{"Player":<25} {"Projected":<12} {"Actual":<12} {"Diff":<10}')
        print('-'*60)
        
        for player in lineup['players']:
            player_id = player['playerID']
            name = player['playerName']
            projected = player['projected_fpts']
            
            # Get actual from slate_df
            player_actual = slate_df[slate_df['playerID'] == player_id]
            if not player_actual.empty:
                actual = player_actual['actual_fpts'].iloc[0]
            else:
                actual = 0
            
            diff = actual - projected
            diff_sign = '+' if diff >= 0 else ''
            
            total_projected += projected
            total_actual += actual
            
            print(f'{name:<25} {projected:<12.2f} {actual:<12.2f} {diff_sign}{diff:<10.2f}')
        
        print('-'*60)
        print(f'{"TOTAL":<25} {total_projected:<12.2f} {total_actual:<12.2f} {total_actual - total_projected:+.2f}')
        
        # Calculate accuracy metrics
        accuracy_pct = (total_actual / total_projected * 100) if total_projected > 0 else 0
        error = abs(total_actual - total_projected)
        
        print(f'\n{"Metrics:":<8}')
        print(f'  Accuracy: {accuracy_pct:.1f}%')
        print(f'  Absolute Error: {error:.2f} pts')
        print(f'  Would have scored: {total_actual:.2f} pts')
        
        # Check if lineup would have cashed (simplified - assumes 50th percentile is ~240 pts)
        cash_line = 240  # Approximate - varies by contest
        would_cash = total_actual >= cash_line
        print(f'  Would cash (>${cash_line}): {"YES ✓" if would_cash else "NO ✗"}')
        
else:
    if not lineups:
        print('No lineups available for evaluation')
    elif 'actual_fpts' not in slate_df.columns:
        print('Note: Actual fantasy points not available (future slate or missing data)')

In [ ]:
# Export lineups
if lineups:
    print('='*80)
    print('EXPORTING LINEUPS')
    print('='*80)
    
    # Determine export directory
    if LINEUP_CONFIG['export_dir'] is not None:
        export_dir = Path(LINEUP_CONFIG['export_dir'])
    else:
        # Use backtest output directory
        export_dir = Path(backtest.run_output_dir) / 'lineups'
    
    export_dir.mkdir(parents=True, exist_ok=True)
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    contest_type = LINEUP_CONFIG['contest_type']
    
    # Export based on configured format
    export_format = LINEUP_CONFIG['export_format']
    
    if export_format in ['csv', 'both']:
        csv_path = export_dir / f'lineups_{contest_type}_{timestamp}.csv'
        generator.export_lineups(lineups, str(csv_path), format='csv')
        print(f'\n✓ CSV exported: {csv_path}')
        
    if export_format in ['json', 'both']:
        json_path = export_dir / f'lineups_{contest_type}_{timestamp}.json'
        generator.export_lineups(lineups, str(json_path), format='json')
        print(f'✓ JSON exported: {json_path}')
    
    # Create summary DataFrame
    lineup_summary = []
    for lineup in lineups:
        lineup_summary.append({
            'lineup_num': lineup['lineup_num'],
            'total_salary': lineup['total_salary'],
            'projected_points': lineup['projected_points'],
            'salary_remaining': lineup['salary_remaining'],
            'pts_per_1k': (lineup['projected_points'] / lineup['total_salary'] * 1000),
            'contest_type': lineup['contest_type'],
            'num_players': len(lineup['players'])
        })
    
    summary_df = pd.DataFrame(lineup_summary)
    
    # Export summary
    summary_path = export_dir / f'lineup_summary_{contest_type}_{timestamp}.csv'
    summary_df.to_csv(summary_path, index=False)
    print(f'✓ Summary exported: {summary_path}')
    
    print(f'\n{"="*80}')
    print('EXPORT COMPLETE')
    print('='*80)
    print(f'\nAll files saved to: {export_dir}')
    print(f'\nLineup Summary:')
    print(summary_df.to_string(index=False))
    
else:
    print('No lineups to export')

In [ ]:
# Display lineup details
if lineups:
    print('='*80)
    print('LINEUP DETAILS')
    print('='*80)
    
    for i, lineup in enumerate(lineups, 1):
        print(f'\n{"="*60}')
        print(f'LINEUP #{i}')
        print(f'{"="*60}')
        print(f'Total Salary: ${lineup["total_salary"]:,}')
        print(f'Projected Points: {lineup["projected_points"]:.2f}')
        print(f'Salary Remaining: ${lineup["salary_remaining"]:,}')
        print(f'Contest Type: {lineup["contest_type"]}')
        
        print(f'\n{"Position":<8} {"Player":<25} {"Team":<5} {"Salary":<10} {"Proj. Pts":<10}')
        print('-'*60)
        
        for player in lineup['players']:
            pos = player.get('position', 'UTIL')
            name = player.get('playerName', 'Unknown')
            team = player.get('team', 'N/A')
            salary = player.get('salary', 0)
            proj_pts = player.get('projected_fpts', 0)
            
            print(f'{pos:<8} {name:<25} {team:<5} ${salary:<9,} {proj_pts:<10.2f}')
        
        # Calculate value metrics
        total_salary = lineup['total_salary']
        total_proj = lineup['projected_points']
        pts_per_k = (total_proj / total_salary * 1000) if total_salary > 0 else 0
        
        print(f'\n{"Metrics:":<8}')
        print(f'  Points per $1K: {pts_per_k:.3f}')
        print(f'  Salary utilization: {(total_salary/50000)*100:.1f}%')
        
        if i < len(lineups):
            print()  # Extra spacing between lineups
            
else:
    print('No lineups to display')

In [ ]:
# Generate optimal lineups
from src.optimization.lineup_generator import LineupGenerator

if slate_df is not None and len(slate_df) > 0:
    print('='*80)
    print('GENERATING OPTIMAL LINEUPS')
    print('='*80)
    
    try:
        # Initialize lineup generator with contest config
        generator = LineupGenerator(contest_config_path=contest_config_file)
        
        # Override num_lineups from config
        generator.contest_config['optimization_settings']['num_lineups'] = LINEUP_CONFIG['num_lineups']
        
        print(f'\nContest: {generator.contest_config["name"]}')
        print(f'Description: {generator.contest_config["description"]}')
        print(f'Strategy: {generator.contest_config["optimization_settings"]["strategy"]}')
        
        # Generate lineups
        print(f'\nGenerating {LINEUP_CONFIG["num_lineups"]} lineup(s)...')
        lineups = generator.generate_lineups(
            predictions_df=slate_df,
            dfs_salaries_df=None  # Already merged
        )
        
        if lineups:
            print(f'\n✓ Successfully generated {len(lineups)} lineup(s)')
        else:
            print('\nWARNING: No lineups generated')
            
    except Exception as e:
        print(f'\nERROR generating lineups: {e}')
        import traceback
        traceback.print_exc()
        lineups = []
else:
    print('ERROR: No slate data available for lineup generation')
    lineups = []

In [ ]:
# Prepare slate data for lineup optimization
from src.data.storage.parquet_storage import ParquetStorage

if 'error' not in results:
    # Get target date
    if LINEUP_CONFIG['target_date'] is not None:
        target_date = str(LINEUP_CONFIG['target_date'])
    else:
        # Use most recent slate from results
        target_date = str(results['daily_results']['date'].max())
    
    print(f'Loading slate data for date: {target_date}')
    
    # Filter predictions for target date
    slate_predictions = all_predictions_df[all_predictions_df['date'] == int(target_date)].copy()
    
    print(f'  Found {len(slate_predictions)} player predictions')
    
    # Load DFS salaries for the slate
    storage = ParquetStorage()
    try:
        dfs_salaries = storage.load('dfs_salaries', filters={'date': target_date})
        print(f'  Loaded {len(dfs_salaries)} salary records')
        
        # Merge predictions with salaries
        slate_df = slate_predictions.merge(
            dfs_salaries[['playerID', 'salary', 'pos', 'team', 'gameInfo', 'status']],
            on='playerID',
            how='inner',
            suffixes=('', '_salary')
        )
        
        # Use salary position if available
        if 'pos_salary' in slate_df.columns and slate_df['pos_salary'].notna().any():
            slate_df['position'] = slate_df['pos_salary']
            slate_df.drop('pos_salary', axis=1, inplace=True)
        elif 'pos' in slate_df.columns:
            slate_df.rename(columns={'pos': 'position'}, inplace=True)
        
        print(f'  Merged data: {len(slate_df)} players ready for optimization')
        
        # Display sample
        print('\nSample slate data:')
        display_cols = ['playerName', 'team', 'position', 'salary', 'predicted_fpts', 'status']
        available_cols = [col for col in display_cols if col in slate_df.columns]
        print(slate_df.nlargest(10, 'predicted_fpts')[available_cols].to_string(index=False))
        
    except Exception as e:
        print(f'ERROR loading DFS salaries: {e}')
        print('Continuing with predictions only (no salary data)')
        slate_df = slate_predictions.copy()
        
else:
    print('ERROR: No backtest results available. Run backtest first.')
    slate_df = None

In [ ]:
# Lineup generation configuration
LINEUP_CONFIG = {
    'contest_type': 'cash_game',  # Options: 'cash_game', 'gpp_tournament', 'single_entry', 'multi_entry'
    'num_lineups': 3,              # Number of lineups to generate
    'target_date': None,           # None = use most recent slate, or specify YYYYMMDD
    'export_format': 'both',       # 'csv', 'json', or 'both'
    'export_dir': None             # None = use backtest output dir
}

# Contest config file mapping
CONTEST_CONFIG_FILES = {
    'cash_game': 'cash_game.json',
    'gpp_tournament': 'gpp_tournament.json',
    'single_entry': 'single_entry.json',
    'multi_entry': 'multi_entry.json'
}

contest_config_file = CONTEST_CONFIG_FILES.get(LINEUP_CONFIG['contest_type'], 'cash_game.json')

print('Lineup Generation Configuration:')
print(f'  Contest Type: {LINEUP_CONFIG["contest_type"]}')
print(f'  Config File: {contest_config_file}')
print(f'  Number of Lineups: {LINEUP_CONFIG["num_lineups"]}')
print(f'  Target Date: {LINEUP_CONFIG["target_date"] or "Most recent slate"}')
print(f'  Export Format: {LINEUP_CONFIG["export_format"]}')